# Heritage Short: ACE-Step 1.5 score candidates (T4)
Pushed by `pipeline/music.py` via `pipeline/kaggle_job.py`. The JOBS cell lists the candidates:
caption (instruments + mood only), and bpm / key / time signature / duration as separate metadata.
Instrumental cues use the `[Instrumental]` lyric tag. Precision modes (`config/models.json: music.mode`, FAILURES.md #16-18):
- `gpu-mixed`: diffusion transformer + text encoder in float16 (fits the T4 with model offload), audio VAE in float32;
- `cpu`: everything float32 on the CPU (slow, numerically safe, no GPU quota);
- `gpu-bf16`: bfloat16 on a GPU that supports it (e.g. NVIDIA L4): fp32's range, so no fp16 overflow.
Whether the latents reaching the VAE are finite is logged, to locate any overflow.
Writes `out/<cue>_c<n>.wav` and `timing.jsonl` to /kaggle/working.

In [ ]:
# @JOBS (replaced at push time)
JOBS = {"job_id": "local-test", "config": {}, "jobs": []}

In [ ]:
import json, os, sys, time, glob, hashlib, subprocess, traceback
T0 = time.time()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/out", exist_ok=True)
_log = open(f"{OUT}/timing.jsonl", "a")
def log(**kw):
    """One JSON line per attempt; pipeline/kaggle_job.py copies these into runs.jsonl."""
    _log.write(json.dumps({"job_id": JOBS["job_id"], **kw}) + "\n"); _log.flush(); print(kw)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.39.0", "soundfile"], check=True)

In [ ]:
import numpy as np, soundfile as sf, torch
from diffusers import AceStepPipeline

cfg = JOBS["config"]
MODE = cfg.get("mode", "gpu-mixed")
DEV = "cpu" if MODE == "cpu" else "cuda"
if MODE == "gpu-bf16" and not torch.cuda.is_bf16_supported():
    log(id=None, event="setup", seconds=round(time.time() - T0, 1), gpu=torch.cuda.get_device_name(0), mode=MODE,
        error="this GPU has no bfloat16")
    raise SystemExit("no bfloat16 on " + torch.cuda.get_device_name(0))
DTYPES = {"cpu": torch.float32, "gpu-bf16": torch.bfloat16, "gpu-mixed": torch.float16}
pipe = AceStepPipeline.from_pretrained(cfg["model"], torch_dtype=DTYPES[MODE])
LAT = {}
if MODE == "gpu-mixed":
    pipe.vae.to(torch.float32)
    _decode = pipe.vae.decode
    def decode_fp32(z, *a, **k):              # cast latents up; record whether the transformer overflowed
        LAT["finite"] = bool(torch.isfinite(z).all())
        return _decode(z.to(torch.float32), *a, **k)
    pipe.vae.decode = decode_fp32
    pipe.enable_model_cpu_offload()
elif MODE == "gpu-bf16":
    pipe.enable_model_cpu_offload()   # bf16 has fp32's exponent range: no fp16-style overflow
else:
    torch.set_num_threads(os.cpu_count() or 4)
log(id=None, event="setup", seconds=round(time.time() - T0, 1),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", torch=torch.__version__, mode=MODE)

In [ ]:
for job in JOBS["jobs"]:
    if DEV == "cuda":
        torch.cuda.reset_peak_memory_stats()
    LAT.clear()
    t = time.time()
    try:
        audio = pipe(prompt=job["caption"], lyrics="[Instrumental]" if job["instrumental"] else "",
                     audio_duration=float(job["duration_s"]), bpm=job["bpm"], keyscale=job["keyscale"],
                     timesignature=job["timesignature"], num_inference_steps=job["steps"], shift=job["shift"],
                     generator=torch.Generator("cpu").manual_seed(job["seed"])).audios
        wav = audio[0].T.cpu().float().numpy()          # [samples, channels]
        if not np.isfinite(wav).all() or float(np.abs(wav).max()) < 1e-3:
            raise RuntimeError(f"NaN or silent audio ({MODE}; latents finite: {LAT.get('finite')})")
        sf.write(f"{OUT}/out/{job['id']}.wav", wav, pipe.sample_rate)
        log(id=job["id"], status="ok", attempt=1, seed=job["seed"], gpu_seconds=round(time.time() - t, 1) if DEV == "cuda" else 0,
            cpu_seconds=round(time.time() - t, 1), duration_s=round(len(wav) / pipe.sample_rate, 2), file=f"out/{job['id']}.wav",
            mode=MODE, latents_finite=LAT.get("finite"),
            peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEV == "cuda" else 0)
    except Exception as e:
        log(id=job["id"], status="error", attempt=1, seed=job["seed"], gpu_seconds=round(time.time() - t, 1) if DEV == "cuda" else 0,
            error=repr(e)[:400], mode=MODE, latents_finite=LAT.get("finite"))
        traceback.print_exc()
log(id=None, event="done", seconds=round(time.time() - T0, 1))